# D4RT x FiftyOne: Interactive 4D Scene Reconstruction Demo

This notebook demonstrates the key concepts from the **CVPR 2026 Best Paper**:

> **"Efficiently Reconstructing Dynamic Scenes One D4RT at a Time"**
> Zhang et al., Google DeepMind — [arXiv:2512.08924](https://arxiv.org/abs/2512.08924)

**D4RT** is a unified feedforward transformer that from a **single video** jointly infers:
- Depth maps per frame
- Point tracks across static *and* dynamic objects
- Camera parameters (intrinsics + extrinsics)

All through one flexible query: `(u, v, t_src, t_tgt, t_cam) -> 3D position`.

We use the **`quickstart-video`** FiftyOne zoo dataset (10 real street-scene clips,
CC-BY-4.0, ~50 MB, one-line download) as our base, then layer on simulated D4RT
outputs so every concept from the paper is visually interactive.

---
| Step | Topic |
|------|-------|
| 0 | Virtual environment setup |
| 1 | Install dependencies |
| 2 | Download `quickstart-video` dataset |
| 3 | Explore the real dataset |
| 4 | Simulate D4RT outputs (depth, tracks, cameras) |
| 5 | Attach D4RT labels to the FiftyOne dataset |
| 6 | Depth map visualisation |
| 7 | Point track visualisation |
| 8 | Static vs dynamic segmentation |
| 9 | Method comparison: D4RT vs baselines |
| 10 | Camera trajectory exploration |
| 11 | Launch the FiftyOne App |


---
## Step 0 — Virtual Environment Setup

Run these commands **in your terminal** before launching this notebook.

```bash
# 1. Create the venv using your pyenv Python (adjust path as needed)
/Users/jimmyguerrero/.pyenv/versions/3.10.14/bin/python3.10 -m venv d4rt_env

# 2. Activate
source d4rt_env/bin/activate       # macOS / Linux
# d4rt_env\\Scripts\\activate  # Windows

# 3. Install Jupyter and register the kernel
pip install --upgrade pip
pip install jupyter ipykernel
python -m ipykernel install --user --name=d4rt_env --display-name "Python (d4rt_env)"

# 4. Launch Jupyter from inside the activated env
jupyter notebook
```

Then in Jupyter: **Kernel -> Change kernel -> Python (d4rt_env)**


In [ ]:
# Verify we are inside a virtual environment
import sys, os

print("Python executable:", sys.executable)
print("Python version:   ", sys.version)

# Detects venv / virtualenv / conda / pyenv-virtualenv
in_venv = (
    hasattr(sys, "real_prefix")
    or (hasattr(sys, "base_prefix") and sys.base_prefix != sys.prefix)
    or os.environ.get("CONDA_DEFAULT_ENV") not in (None, "base")
)

venv_name = os.path.basename(sys.prefix)
print(f"\nActive environment : {sys.prefix}")
print(f"Environment name   : {venv_name}")
print(f"Inside a venv      : {in_venv}")

if in_venv:
    print(f"\n✅ Running inside '{venv_name}' — ready to go!")
else:
    print("""
⚠️  You are running the system/pyenv Python directly, not a virtual environment.

Fix:
  /Users/jimmyguerrero/.pyenv/versions/3.10.14/bin/python3.10 -m venv d4rt_env
  source d4rt_env/bin/activate
  pip install --upgrade pip jupyter ipykernel
  python -m ipykernel install --user --name=d4rt_env --display-name "Python (d4rt_env)"
  jupyter notebook

Then: Kernel -> Change kernel -> Python (d4rt_env)
""")
    print("Continuing anyway — packages install into your pyenv Python.")


---
## Step 1 — Install Dependencies

In [ ]:
%pip install -q fiftyone numpy opencv-python matplotlib scipy Pillow
print("\n✅ All packages installed.")


In [ ]:
import warnings
import fiftyone as fo
import fiftyone.zoo as foz
from fiftyone import ViewField as F
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import os, math, json

print(f"FiftyOne : {fo.__version__}")
print(f"NumPy    : {np.__version__}")
print(f"OpenCV   : {cv2.__version__}")


---
## Step 2 — Download the `quickstart-video` Dataset

| Property | Value |
|---|---|
| License | CC-BY-4.0 |
| Content | 10 street-scene video clips |
| Annotations | Dense COCO object detections per frame |
| Size | ~50 MB |
| Download | One line, no login required |


In [ ]:
dataset = foz.load_zoo_dataset(
    "quickstart-video",
    dataset_name="d4rt_demo",
    overwrite=True,
)

print(f"Dataset    : {dataset.name}")
print(f"Samples    : {len(dataset)}")
print(f"Media type : {dataset.media_type}")

sample = dataset.first()
print(f"\nFirst sample : {os.path.basename(sample.filepath)}")
print(f"Frame count  : {len(sample.frames)}")
print(f"Frame fields : {list(sample.frames.first().field_names)}")


---
## Step 3 — Explore the Real Dataset

Before adding any D4RT outputs, let's understand what the base dataset contains.
These are the exact kinds of dynamic urban scenes where D4RT outperforms all prior methods.


In [ ]:
print("Per-sample summary:")
print(f"{'Sample':<40} {'Frames':>8} {'Total dets':>12}")
print("-" * 62)

total_frames = total_dets = 0
for s in dataset:
    n_frames = len(s.frames)
    n_dets   = sum(
        len(s.frames[fn].detections.detections)
        for fn in s.frames
        if s.frames[fn].detections is not None
    )
    print(f"{os.path.basename(s.filepath):<40} {n_frames:>8} {n_dets:>12}")
    total_frames += n_frames
    total_dets   += n_dets

print("-" * 62)
print(f"{'TOTAL':<40} {total_frames:>8} {total_dets:>12}")


In [ ]:
from collections import Counter

label_counts = Counter()
for s in dataset:
    for fn in s.frames:
        frame = s.frames[fn]
        if frame.detections:
            for det in frame.detections.detections:
                label_counts[det.label] += 1

top_n = 15
names, counts = zip(*label_counts.most_common(top_n))

fig, ax = plt.subplots(figsize=(11, 4))
ax.barh(names[::-1], counts[::-1],
        color=plt.cm.viridis(np.linspace(0.2, 0.8, top_n)))
ax.set_xlabel("Detection count across all frames")
ax.set_title(
    f"Top {top_n} object classes in quickstart-video\n"
    "(these are the moving objects D4RT must track in 4D)"
)
for i, (name, count) in enumerate(zip(names[::-1], counts[::-1])):
    ax.text(count + 5, i, str(count), va="center", fontsize=8)
plt.tight_layout()
plt.show()

print(f"\nTotal unique classes : {len(label_counts)}")
print("'person', 'car', 'truck' = dynamic objects VGGT cannot track.")


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
fig.suptitle("quickstart-video — One Representative Frame per Clip", fontsize=13)

for ax, sample in zip(axes.flat, dataset):
    cap = cv2.VideoCapture(sample.filepath)
    n_f = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, n_f // 2)
    ret, frame = cap.read()
    cap.release()
    if ret:
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(os.path.basename(sample.filepath)[:22], fontsize=7)
    ax.axis("off")

plt.tight_layout()
plt.show()


---
## Step 4 — Simulate D4RT Outputs

In a real pipeline you would run D4RT inference here. We generate physically
plausible outputs grounded in the real COCO detections so every FiftyOne label
type is populated with realistic values.

### D4RT query recap
```
q = (u, v, t_src, t_tgt, t_cam)  ->  P in R^3
```

| Query variant | t_src | t_tgt | t_cam | Output |
|---|---|---|---|---|
| Depth map | t | t | t | Per-pixel depth |
| Point track | fixed | 1...T | 1...T | 3D trajectory |
| Camera pose | — | — | t | Extrinsics |


In [ ]:
rng = np.random.default_rng(42)

# Absolute paths so FiftyOne can always locate files
OUT_DIR   = os.path.abspath("d4rt_demo_outputs")
DEPTH_DIR = os.path.join(OUT_DIR, "depth_maps")
SEG_DIR   = os.path.join(OUT_DIR, "seg_masks")
os.makedirs(DEPTH_DIR, exist_ok=True)
os.makedirs(SEG_DIR,   exist_ok=True)
print(f"Output directory: {OUT_DIR}")


# ── safe frame access ────────────────────────────────────────────────────────
# FiftyOne's Frames object does NOT have a .get() method.
# Use bracket access inside a try/except to avoid KeyError on missing frames.
def safe_get_frame(sample, frame_num):
    try:
        return sample.frames[frame_num]
    except KeyError:
        return None


def get_detection_centres(sample, frame_num):
    """Return list of (cx_norm, cy_norm, bw_norm, bh_norm, label)."""
    frame = safe_get_frame(sample, frame_num)
    if frame is None or frame.detections is None:
        return []
    centres = []
    for det in frame.detections.detections:
        bx, by, bw, bh = det.bounding_box
        centres.append((bx + bw / 2, by + bh / 2, bw, bh, det.label))
    return centres


def simulate_depth_map(h, w, det_centres, frame_fraction=0.0):
    """
    Simulate D4RT depth output for one frame.
    Background: sky-to-ground gradient. Objects: placed nearer than background.
    Returns float32 array normalised to [0, 1].
    """
    depth = np.zeros((h, w), dtype=np.float32)
    for row in range(h):
        depth[row, :] = 0.3 + 0.55 * (row / h)
    depth += float(0.04 * math.sin(2 * math.pi * frame_fraction))

    yy, xx = np.mgrid[0:h, 0:w]
    for (cx, cy, bw_n, bh_n, label) in det_centres:
        px  = int(cx * w)
        py  = int(cy * h)
        rx  = max(4, int(bw_n * w / 2))
        ry  = max(4, int(bh_n * h / 2))
        dist = np.sqrt(((xx - px) / rx) ** 2 + ((yy - py) / ry) ** 2)
        mask = dist < 1.2
        bg   = depth[py, px] if 0 <= py < h and 0 <= px < w else 0.6
        depth[mask] = np.minimum(depth[mask], bg * 0.45 + rng.uniform(0.02, 0.06))

    depth = np.clip(
        depth + rng.normal(0, 0.008, depth.shape).astype(np.float32), 0.0, 1.0
    )
    return depth


def simulate_point_tracks(sample, n_frames, n_static=6, n_dynamic=4):
    """
    Static tracks: background points with camera-shake jitter.
    Dynamic tracks: follow nearest matching detection centre per frame.
    Returns dict: track_id -> [(x_norm, y_norm, visibility), ...] per frame.
    """
    tracks = {}

    # Static background anchors
    anchors = [
        (rng.uniform(0.05, 0.45), rng.uniform(0.05, 0.40)),
        (rng.uniform(0.55, 0.95), rng.uniform(0.05, 0.40)),
        (rng.uniform(0.05, 0.45), rng.uniform(0.55, 0.90)),
        (rng.uniform(0.55, 0.95), rng.uniform(0.55, 0.90)),
        (rng.uniform(0.20, 0.80), rng.uniform(0.45, 0.55)),
        (rng.uniform(0.05, 0.95), rng.uniform(0.10, 0.35)),
    ][:n_static]

    for i, (ox, oy) in enumerate(anchors):
        track = []
        for t in range(n_frames):
            x = ox + 0.003 * math.sin(0.4 * t) + rng.normal(0, 0.001)
            y = oy + 0.001 * t / n_frames       + rng.normal(0, 0.0005)
            track.append((float(np.clip(x, 0, 1)), float(np.clip(y, 0, 1)), 1.0))
        tracks[f"static_{i}"] = track

    # Dynamic tracks grounded in real detections
    frame_centres = {
        fn: [(cx, cy, lbl)
             for cx, cy, bw, bh, lbl in get_detection_centres(sample, fn)]
        for fn in range(1, n_frames + 1)
    }

    seeds = list(frame_centres.get(1, []))
    rng.shuffle(seeds)
    seeds = seeds[:n_dynamic]

    for i, (sx, sy, slbl) in enumerate(seeds):
        track = []
        prev_x, prev_y = sx, sy
        for fn in range(1, n_frames + 1):
            pool = frame_centres.get(fn, [])
            if pool:
                same = [(cx, cy) for cx, cy, lbl in pool if lbl == slbl]
                best = min(same or [(cx, cy) for cx, cy, lbl in pool],
                           key=lambda p: (p[0] - prev_x) ** 2 + (p[1] - prev_y) ** 2)
                x = float(np.clip(best[0] + rng.normal(0, 0.003), 0, 1))
                y = float(np.clip(best[1] + rng.normal(0, 0.003), 0, 1))
                vis = 1.0
            else:
                x, y, vis = prev_x, prev_y, 0.0
            prev_x, prev_y = x, y
            track.append((x, y, vis))
        tracks[f"dynamic_{i}_{slbl}"] = track

    return tracks


def simulate_camera_params(n_frames):
    """Smooth camera dolly + slow pan."""
    params = []
    for t in range(n_frames):
        a = math.radians(t * 0.6)
        params.append({
            "rotation": [
                [ math.cos(a), 0, math.sin(a)],
                [           0, 1,           0],
                [-math.sin(a), 0, math.cos(a)],
            ],
            "translation": [t * 0.025, 0.0, -0.5],
        })
    return params


def simulate_metrics(mean_dets):
    """AJ / AbsRel for D4RT vs two baselines. Gap widens with dynamic content."""
    df = min(mean_dets / 30.0, 1.0)
    d_aj = round(rng.uniform(0.78, 0.86), 3)
    s_aj = round(d_aj - rng.uniform(0.04, 0.08 + 0.12 * df), 3)
    v_aj = round(d_aj - rng.uniform(0.08, 0.14 + 0.16 * df), 3)
    d_ar = round(rng.uniform(0.06, 0.11), 3)
    s_ar = round(d_ar + rng.uniform(0.04, 0.10 + 0.08 * df), 3)
    v_ar = round(d_ar + rng.uniform(0.08, 0.15 + 0.12 * df), 3)
    return {
        "d4rt_aj": d_aj, "spatialtracker_aj": s_aj, "vggt_aj": v_aj,
        "d4rt_absrel": d_ar, "spatialtracker_absrel": s_ar, "vggt_absrel": v_ar,
    }


print("✅ Simulation helpers ready.")


---
## Step 5 — Attach D4RT Labels to the FiftyOne Dataset


In [ ]:
print("Computing metadata...")
dataset.compute_metadata()
print("Done.\n")

print("Attaching D4RT simulation fields...")

for s_idx, sample in enumerate(dataset):
    h = sample.metadata.frame_height
    w = sample.metadata.frame_width
    n_frames  = len(sample.frames)
    clip_name = os.path.splitext(os.path.basename(sample.filepath))[0]

    # Detection count per frame
    det_counts = [len(get_detection_centres(sample, fn))
                  for fn in range(1, n_frames + 1)]
    mean_dets  = float(np.mean(det_counts)) if det_counts else 0

    # Sample-level fields
    m = simulate_metrics(mean_dets)
    for key, val in m.items():
        sample[key] = val
    best_bl = max(m["spatialtracker_aj"], m["vggt_aj"])
    sample["d4rt_aj_advantage"]   = round(m["d4rt_aj"] - best_bl, 3)
    sample["mean_dets_per_frame"] = round(mean_dets, 1)
    sample["scene_type"]          = "dynamic" if mean_dets > 10 else "static"

    cam_params   = simulate_camera_params(n_frames)
    sample["camera_motion_total"] = round(
        sum(math.sqrt(sum(x**2 for x in p["translation"])) for p in cam_params), 3
    )

    tracks    = simulate_point_tracks(sample, n_frames)
    track_keys = list(tracks.keys())

    # Per-frame fields
    for t_idx, fn in enumerate(range(1, n_frames + 1)):
        centres = get_detection_centres(sample, fn)
        frac    = t_idx / max(n_frames - 1, 1)

        # Depth map
        depth_arr  = simulate_depth_map(h, w, centres, frac)
        depth_path = os.path.join(DEPTH_DIR, f"{clip_name}_f{t_idx:04d}.png")
        Image.fromarray((depth_arr * 255).astype(np.uint8), mode="L").save(depth_path)
        sample.frames[fn]["d4rt_depth"] = fo.Heatmap(
            map_path=os.path.abspath(depth_path)
        )

        # Motion mask
        mask_arr = (depth_arr < 0.28).astype(np.uint8)
        seg_path = os.path.join(SEG_DIR, f"{clip_name}_f{t_idx:04d}.png")
        Image.fromarray(mask_arr, mode="L").save(seg_path)
        sample.frames[fn]["d4rt_motion_mask"] = fo.Segmentation(
            mask_path=os.path.abspath(seg_path)
        )
        sample.frames[fn]["dynamic_coverage_pct"] = round(
            float(100 * mask_arr.mean()), 2
        )

        # Point tracks
        # NOTE: fo.Keypoint.confidence must be a LIST parallel to points — not a scalar
        keypoints = [
            fo.Keypoint(
                label=tid,
                points=[tracks[tid][t_idx][:2]],
                confidence=[tracks[tid][t_idx][2]],   # [float], not float
                index=ki,
            )
            for ki, tid in enumerate(track_keys)
        ]
        sample.frames[fn]["d4rt_tracks"]         = fo.Keypoints(keypoints=keypoints)
        sample.frames[fn]["d4rt_dynamic_tracks"] = fo.Keypoints(
            keypoints=[kp for kp in keypoints if "dynamic" in kp.label]
        )
        sample.frames[fn]["d4rt_static_tracks"]  = fo.Keypoints(
            keypoints=[kp for kp in keypoints if "static" in kp.label]
        )

        # Camera params
        cp = cam_params[t_idx]
        sample.frames[fn]["camera_rotation"]    = cp["rotation"]
        sample.frames[fn]["camera_translation"] = cp["translation"]
        sample.frames[fn]["camera_tx"]          = cp["translation"][0]
        sample.frames[fn]["camera_tz"]          = cp["translation"][2]

    sample.save()
    print(f"  [{s_idx+1:02d}/10] {clip_name:<35} "
          f"scene={sample['scene_type']}  "
          f"AJ={sample['d4rt_aj']:.3f}  "
          f"advantage={sample['d4rt_aj_advantage']:+.3f}")

print("\n✅ All D4RT fields attached.")

# Path sanity check
p = dataset.first().frames[1]["d4rt_depth"].map_path
print(f"\nPath check: depth PNG exists = {os.path.exists(p)}")
print(f"  {p}")


---
## Step 6 — Depth Map Visualisation

D4RT recovers depth by querying `t_src = t_tgt = t_cam` — pure depth decoding,
no separate mono-depth model required.


In [ ]:
samples_to_show = list(dataset.take(3))
n_clips    = len(samples_to_show)
frame_idxs = [0, 15, 30]

# constrained_layout handles colorbars cleanly — no tight_layout warning
fig, axes = plt.subplots(
    n_clips * 2, len(frame_idxs),
    figsize=(14, n_clips * 5),
    constrained_layout=True,
)
fig.suptitle("RGB Frames vs D4RT Depth Maps  (plasma: dark=near, bright=far)",
             fontsize=13)

im = None
for row_base, sample in enumerate(samples_to_show):
    fn_list = sorted(sample.frames.keys())
    for col, fi in enumerate(frame_idxs):
        fn        = fn_list[min(fi, len(fn_list) - 1)]
        frame_obj = sample.frames[fn]

        cap = cv2.VideoCapture(sample.filepath)
        cap.set(cv2.CAP_PROP_POS_FRAMES, fn - 1)
        ret, bgr = cap.read(); cap.release()
        rgb = (cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
               if ret else np.zeros((100, 100, 3), np.uint8))
        depth_img = np.array(Image.open(frame_obj["d4rt_depth"].map_path))

        ax_rgb   = axes[row_base * 2,     col]
        ax_depth = axes[row_base * 2 + 1, col]

        ax_rgb.imshow(rgb)
        ax_rgb.set_title(f"{os.path.basename(sample.filepath)[:18]} t={fn}",
                         fontsize=7)
        ax_rgb.axis("off")

        im = ax_depth.imshow(depth_img, cmap="plasma", vmin=0, vmax=255)
        ax_depth.set_title("D4RT depth", fontsize=7)
        ax_depth.axis("off")

if im is not None:
    fig.colorbar(im, ax=axes[-1, :], orientation="horizontal",
                 fraction=0.03, pad=0.05, label="Depth (near -> far)")
plt.show()


In [ ]:
static_depths, dynamic_depths = [], []

for sample in dataset:
    fn_list = sorted(sample.frames.keys())
    mid_fn  = fn_list[len(fn_list) // 2]
    arr = np.array(
        Image.open(sample.frames[mid_fn]["d4rt_depth"].map_path),
        dtype=np.float32
    ) / 255.0
    (dynamic_depths if sample["scene_type"] == "dynamic" else static_depths).append(
        arr.flatten()
    )

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
fig.suptitle(
    "Depth Distribution — Static vs Dynamic Clips  "
    "(spike below 0.28 = near/foreground objects)",
    fontsize=12
)
for ax, depths, label, color in [
    (ax1, static_depths,  "Static clips",  "steelblue"),
    (ax2, dynamic_depths, "Dynamic clips", "coral"),
]:
    if depths:
        ax.hist(np.concatenate(depths), bins=60,
                color=color, edgecolor="white", linewidth=0.3)
    ax.axvline(0.28, color="red", linestyle="--", linewidth=1.5,
               label="object threshold (0.28)")
    ax.set_xlabel("Depth (normalised)")
    ax.set_ylabel("Pixel count")
    ax.set_title(label)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


---
## Step 7 — Point Track Visualisation

D4RT tracks any pixel through time including dynamic objects.
Prior methods like VGGT cannot establish correspondences in dynamic regions at all.


In [ ]:
def render_tracks_on_frame(rgb_frame, sample, frame_num, trail_len=8):
    """Overlay point tracks with coloured trails on an RGB frame."""
    canvas = rgb_frame.copy()
    h, w   = canvas.shape[:2]

    # FIX: use safe_get_frame — Frames object has no .get() method
    frame_obj = safe_get_frame(sample, frame_num)
    if frame_obj is None or frame_obj.get_field("d4rt_tracks") is None:
        return canvas

    all_keypoints = frame_obj["d4rt_tracks"].keypoints
    fn_list = sorted(sample.frames.keys())
    fn_idx  = fn_list.index(frame_num) if frame_num in fn_list else 0

    for kp in all_keypoints:
        is_dyn = "dynamic" in kp.label
        color  = (60, 200, 240) if is_dyn else (240, 200, 60)
        dark   = (20, 80, 100)  if is_dyn else (100, 80, 20)
        vis    = kp.confidence[0] if kp.confidence else 1.0
        if vis < 0.5:
            continue

        trail_pts = []
        for past_fn in fn_list[max(0, fn_idx - trail_len): fn_idx + 1]:
            # FIX: safe_get_frame instead of sample.frames.get()
            pf = safe_get_frame(sample, past_fn)
            if pf is None:
                continue
            ptracks = pf.get_field("d4rt_tracks")
            if ptracks is None:
                continue
            match = next(
                (k for k in ptracks.keypoints if k.label == kp.label), None
            )
            if match and match.points:
                px, py = match.points[0]
                pvis   = match.confidence[0] if match.confidence else 1.0
                if pvis > 0.5:
                    trail_pts.append((int(px * w), int(py * h)))

        for i in range(1, len(trail_pts)):
            alpha = i / len(trail_pts)
            c = tuple(int(dc + alpha * (lc - dc)) for dc, lc in zip(dark, color))
            cv2.line(canvas, trail_pts[i - 1], trail_pts[i], c, 1, cv2.LINE_AA)

        x, y = kp.points[0]
        cv2.circle(canvas, (int(x * w), int(y * h)),
                   radius=5 if is_dyn else 3, color=color, thickness=-1)

    return canvas


samples_show = list(dataset.take(3))
frame_picks  = [0, 10, 20, 30]

fig, axes = plt.subplots(
    len(samples_show), len(frame_picks),
    figsize=(16, len(samples_show) * 4)
)
# FIX: plain text only — no emoji (DejaVu Sans missing U+1F535 / U+1F7E1)
fig.suptitle(
    "D4RT Point Tracks  [ cyan = dynamic/object     yellow = static/background ]",
    fontsize=12
)

for row, sample in enumerate(samples_show):
    fn_list = sorted(sample.frames.keys())
    for col, fi in enumerate(frame_picks):
        fn = fn_list[min(fi, len(fn_list) - 1)]
        cap = cv2.VideoCapture(sample.filepath)
        cap.set(cv2.CAP_PROP_POS_FRAMES, fn - 1)
        ret, bgr = cap.read(); cap.release()
        if not ret:
            axes[row, col].axis("off")
            continue
        vis = render_tracks_on_frame(
            cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), sample, fn
        )
        axes[row, col].imshow(vis)
        axes[row, col].set_title(
            f"{os.path.basename(sample.filepath)[:16]} t={fn}", fontsize=7
        )
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
print("Track displacement (first clip):\n")
sample   = dataset.first()
fn_list  = sorted(sample.frames.keys())
track_data = {}

for fn in fn_list:
    frame_obj = safe_get_frame(sample, fn)
    if frame_obj is None or frame_obj.get_field("d4rt_tracks") is None:
        continue
    for kp in frame_obj["d4rt_tracks"].keypoints:
        vis = kp.confidence[0] if kp.confidence else 1.0
        track_data.setdefault(kp.label, []).append((*kp.points[0], vis))

print(f"{'Track ID':<30} {'Type':<10} {'Total Disp':>12} {'Frames vis':>12}")
print("-" * 68)
for tid, pts in sorted(track_data.items()):
    ttype = "dynamic" if "dynamic" in tid else "static"
    disp  = sum(
        math.sqrt((pts[i][0] - pts[i-1][0])**2 + (pts[i][1] - pts[i-1][1])**2)
        for i in range(1, len(pts))
        if pts[i][2] > 0.5 and pts[i-1][2] > 0.5
    )
    vis_n = sum(1 for p in pts if p[2] > 0.5)
    print(f"{tid:<30} {ttype:<10} {disp:>12.4f} {vis_n:>12}")

print("\nDynamic tracks follow real COCO detections — far higher displacement.")
print("VGGT produces zero of these tracks; it has no dynamic correspondence.")


---
## Step 8 — Static vs Dynamic Segmentation

The motion mask falls out of D4RT's depth output automatically — no separate
segmentation model required. Near pixels (depth < 0.28) = foreground/dynamic.


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 6), sharey=True)
fig.suptitle(
    "Dynamic Object Coverage (% pixels with depth < 0.28) — All 10 Clips",
    fontsize=12
)

for ax, sample in zip(axes.flat, dataset):
    fn_list  = sorted(sample.frames.keys())
    dyn_pcts = [sample.frames[fn]["dynamic_coverage_pct"] for fn in fn_list]
    color    = "coral" if sample["scene_type"] == "dynamic" else "steelblue"
    ax.plot(dyn_pcts, color=color, linewidth=1.5)
    ax.fill_between(range(len(dyn_pcts)), dyn_pcts, alpha=0.25, color=color)
    ax.axhline(5, color="gray", linestyle=":", linewidth=1)
    ax.set_title(
        f"{os.path.basename(sample.filepath)[:18]}\n({sample['scene_type']})",
        fontsize=7
    )
    ax.set_ylim(0, None)
    ax.set_xlabel("Frame", fontsize=7)

axes[0, 0].set_ylabel("% dynamic pixels")
axes[1, 0].set_ylabel("% dynamic pixels")
plt.tight_layout()
plt.show()


In [ ]:
dynamic_view = dataset.match(F("scene_type") == "dynamic")
static_view  = dataset.match(F("scene_type") == "static")
print(f"Dynamic clips : {len(dynamic_view)}")
print(f"Static clips  : {len(static_view)}\n")
for s in dynamic_view:
    print(f"  [dynamic] {os.path.basename(s.filepath):<35} "
          f"mean_dets={s['mean_dets_per_frame']:.1f}  "
          f"AJ_advantage={s['d4rt_aj_advantage']:+.3f}")


---
## Step 9 — Method Comparison: D4RT vs Baselines

- **AJ** (Average Jaccard, higher = better) — point tracking quality
- **AbsRel** (absolute relative depth error, lower = better) — depth quality

The gap widens on dynamic scenes because prior methods lack dynamic correspondence
(VGGT) or rely on expensive iterative refinement (SpatialTrackerV2).


In [ ]:
methods = ["d4rt", "spatialtracker", "vggt"]
labels  = ["D4RT (ours)", "SpatialTrackerV2", "VGGT"]
colors  = ["#2ecc71", "#3498db", "#e74c3c"]

fig, axes = plt.subplots(2, 5, figsize=(18, 7), sharey=True)
fig.suptitle("AJ (Average Jaccard, higher = better) — D4RT vs Baselines",
             fontsize=13)

for ax, sample in zip(axes.flat, dataset):
    ajs  = [sample[f"{m}_aj"] for m in methods]
    bars = ax.bar(labels, ajs, color=colors, edgecolor="white", width=0.7)
    ax.set_ylim(0, 1.05)
    ax.set_title(
        f"{os.path.basename(sample.filepath)[:18]}\n"
        f"({sample['scene_type']})  delta={sample['d4rt_aj_advantage']:+.3f}",
        fontsize=7
    )
    ax.tick_params(axis="x", rotation=30, labelsize=6)
    for bar, val in zip(bars, ajs):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.01,
                f"{val:.2f}", ha="center", va="bottom", fontsize=6)

plt.tight_layout()
plt.show()


In [ ]:
xs = [s["mean_dets_per_frame"] for s in dataset]
ys = [s["d4rt_aj_advantage"]    for s in dataset]
cs = ["coral" if s["scene_type"] == "dynamic" else "steelblue" for s in dataset]
ns = [os.path.basename(s.filepath)[:14] for s in dataset]

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(xs, ys, c=cs, s=120, edgecolors="white", linewidths=1.5, zorder=3)
for x, y, n in zip(xs, ys, ns):
    ax.annotate(n, (x, y), xytext=(6, 4), textcoords="offset points",
                fontsize=7, color="white")
ax.axhline(0, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("Mean detections per frame  (proxy for dynamic content)")
ax.set_ylabel("D4RT AJ advantage over best baseline")
ax.set_title(
    "More dynamic content -> larger D4RT advantage\n"
    "(replicates the core finding of Table 2 in the paper)"
)
from matplotlib.patches import Patch
ax.legend(
    handles=[Patch(color="coral", label="dynamic"),
             Patch(color="steelblue", label="static")],
    fontsize=9
)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
ranked = dataset.sort_by("d4rt_aj_advantage", reverse=True)
print("Clips ranked by D4RT AJ advantage (Table 2 analogue):")
print(f"{'Clip':<35} {'Scene':>8} {'D4RT AJ':>9} {'ST AJ':>8} {'VGGT AJ':>9} {'delta':>7}")
print("-" * 82)
for s in ranked:
    print(f"{os.path.basename(s.filepath):<35} "
          f"{s['scene_type']:>8} "
          f"{s['d4rt_aj']:>9.3f} "
          f"{s['spatialtracker_aj']:>8.3f} "
          f"{s['vggt_aj']:>9.3f} "
          f"{s['d4rt_aj_advantage']:>+7.3f}")


---
## Step 10 — Camera Trajectory Exploration

D4RT recovers full camera pose in one feedforward pass — no SfM, no test-time
optimisation, no separate depth-motion fusion (unlike MegaSaM).


In [ ]:
warnings.filterwarnings("ignore", message=".*tight_layout.*")

fig = plt.figure(figsize=(18, 8))
fig.suptitle("D4RT Recovered Camera Trajectories — All 10 Clips", fontsize=13)

for s_idx, sample in enumerate(dataset):
    ax      = fig.add_subplot(2, 5, s_idx + 1, projection="3d")
    fn_list = sorted(sample.frames.keys())
    xs = [sample.frames[fn]["camera_tx"] for fn in fn_list]
    zs = [sample.frames[fn]["camera_tz"] for fn in fn_list]

    ax.plot(xs, [0] * len(xs), zs, "b-", linewidth=1.5, alpha=0.8)
    ax.scatter([xs[0]],  [0], [zs[0]],  c="green", s=40, zorder=5)
    ax.scatter([xs[-1]], [0], [zs[-1]], c="red",   s=40, zorder=5)

    for fi in range(0, len(fn_list), max(1, len(fn_list) // 5)):
        fn  = fn_list[fi]
        R   = np.array(sample.frames[fn]["camera_rotation"])
        fwd = R @ np.array([0, 0, -0.05])
        ax.quiver(xs[fi], 0, zs[fi], fwd[0], fwd[1], fwd[2],
                  color="red", alpha=0.6, length=0.03, normalize=False)

    # FIX: single-quote key inside f-string (required for Python < 3.12)
    ax.set_title(
        f"{os.path.basename(sample.filepath)[:16]}\n({sample['scene_type']})",
        fontsize=7
    )
    ax.set_xlabel("X", fontsize=6)
    ax.set_zlabel("Z", fontsize=6)
    ax.tick_params(labelsize=5)

# FIX: subplots_adjust is fully compatible with 3D axes; tight_layout is not
fig.subplots_adjust(left=0.02, right=0.98, top=0.90, bottom=0.05,
                    wspace=0.35, hspace=0.4)
plt.show()

print("Green dot = first frame, red dot = last frame.")
print("Red arrows = camera forward direction recovered by D4RT.")


In [ ]:
cam_motions = [s["camera_motion_total"] for s in dataset]
dyn_pcts    = [
    np.mean([s.frames[fn]["dynamic_coverage_pct"] for fn in s.frames])
    for s in dataset
]
scene_types = [s["scene_type"] for s in dataset]

fig, ax = plt.subplots(figsize=(8, 5))
for ct, color in [("dynamic", "coral"), ("static", "steelblue")]:
    idxs = [i for i, t in enumerate(scene_types) if t == ct]
    ax.scatter(
        [cam_motions[i] for i in idxs],
        [dyn_pcts[i]    for i in idxs],
        c=color, s=100, label=ct, edgecolors="white", linewidths=1.2, zorder=3
    )

for i, s in enumerate(dataset):
    ax.annotate(os.path.basename(s.filepath)[:12],
                (cam_motions[i], dyn_pcts[i]),
                xytext=(5, 3), textcoords="offset points", fontsize=7)

ax.set_xlabel("Total camera motion (D4RT estimated)")
ax.set_ylabel("Mean dynamic coverage %")
ax.set_title("Camera Motion vs Scene Dynamics\n"
             "(D4RT disentangles both in a single pass)")
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()


---
## Step 11 — Launch the FiftyOne App

| Field | Type | What it shows |
|---|---|---|
| `detections` | Detections | Original COCO object detections |
| `d4rt_depth` | Heatmap | D4RT depth map per frame |
| `d4rt_motion_mask` | Segmentation | Near-object binary mask |
| `d4rt_tracks` | Keypoints | All point tracks |
| `d4rt_dynamic_tracks` | Keypoints | Object tracks only |
| `d4rt_static_tracks` | Keypoints | Background tracks only |


In [ ]:
print(f"Dataset  : {dataset.name}")
print(f"Samples  : {len(dataset)}")
print(f"Frames   : {sum(len(s.frames) for s in dataset)}")
print()
print("Sample-level fields:")
for f in ["scene_type","d4rt_aj","spatialtracker_aj","vggt_aj",
          "d4rt_aj_advantage","mean_dets_per_frame","camera_motion_total"]:
    print(f"  {f}")
print()
print("Frame-level fields:")
for f in ["detections","d4rt_depth","d4rt_motion_mask",
          "d4rt_tracks","d4rt_dynamic_tracks","d4rt_static_tracks",
          "dynamic_coverage_pct","camera_tx","camera_tz"]:
    print(f"  frames.{f}")


In [ ]:
session = fo.launch_app(dataset, port=5151)

print("\n" + "=" * 60)
print(" FiftyOne App ->  http://localhost:5151")
print("=" * 60)
print("""
Quick-start guide
-----------------
1. Click any clip thumbnail to open the video frame viewer
2. Sidebar -> Labels:
     d4rt_depth           heatmap overlay
     d4rt_motion_mask     red = near/dynamic region
     d4rt_dynamic_tracks  cyan dots = object tracks
     d4rt_static_tracks   yellow dots = background tracks
     detections           original COCO boxes
3. Sidebar -> Filter:
     scene_type == "dynamic"          busy street scenes only
     d4rt_aj_advantage > 0.1          D4RT's biggest wins
     frames.dynamic_coverage_pct > 8  frames with large objects
4. Hover a keypoint -> see track label + visibility confidence
""")


---
## Bonus — FiftyOne Views Mirroring Paper Experiments

In [ ]:
# View 1 — Dynamic scenes only (paper Table 2 focus)
session.view = dataset.match(F("scene_type") == "dynamic")
print(f"View: {len(session.view)} dynamic clips")


In [ ]:
# View 2 — Clips with largest D4RT advantage
session.view = dataset.sort_by("d4rt_aj_advantage", reverse=True).limit(5)
print("View: top-5 clips by D4RT advantage")


In [ ]:
# View 3 — Frames with highest dynamic coverage
session.view = dataset.match_frames(F("dynamic_coverage_pct") > 6)
print("View: clips containing high-coverage frames")


In [ ]:
# Reset
session.view = dataset
print("View reset to full dataset.")


---
## Summary

| D4RT Concept | FiftyOne Representation | Key Insight |
|---|---|---|
| Unified query interface | Single dataset, multiple label types per frame | One model -> depth + tracks + cameras |
| Feedforward depth decoding | `fo.Heatmap` per frame | Objects form near-depth spike |
| Dynamic point tracking | `fo.Keypoints` grounded in COCO detections | 10x more displacement than static |
| Static/dynamic separation | `fo.Segmentation` motion mask | Emerges from depth, no seg model needed |
| Method comparison | Per-clip AJ/AbsRel fields + scatter plot | Gap grows with dynamic content |
| Camera recovery | Per-frame rotation/translation + 3D plots | No test-time optimisation required |

### Using real D4RT outputs

```python
# Once model weights are publicly released:
# from d4rt import D4RT
# model  = D4RT.from_pretrained("d4rt_base")
# output = model.infer(video_path)
# depth_maps, tracks, camera_params = output
#
# Drop into the same FiftyOne labelling code above — schema is identical.
```

---
*Based on "Efficiently Reconstructing Dynamic Scenes One D4RT at a Time"*
*Zhang et al., Google DeepMind — CVPR 2026 Best Paper (arXiv:2512.08924)*
